In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [54]:
# Configuration
OUTPUT_DIR = "results"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

In [55]:
# Data Fetching

DATA_PATH = 'data/online_shoppers_intention.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print("First 5 Observations:")
display(df.head())

print("\nData Types and Non-Null Summary:")
df.info()

Dataset Dimensions: 12,330 rows x 18 columns

First 5 Observations:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



Data Types and Non-Null Summary:
<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14 

In [56]:
# Clean data set 

print("Missing values across all features:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values across all features: 0
Duplicate rows: 125


In [57]:
counts = df["Revenue"].value_counts()
pct_purchase = 100 * counts[True] / len(df)
print("\nClass Distribution (Target: Revenue):",pct_purchase)
print(f"  False (No Purchase): {counts[False]:,} ({100 - pct_purchase:.2f}%)")
print(f"  True  (Purchase):    {counts[True]:,} ({pct_purchase:.2f}%)")


Class Distribution (Target: Revenue): 15.474452554744525
  False (No Purchase): 10,422 (84.53%)
  True  (Purchase):    1,908 (15.47%)


In [58]:
# display the graph of Purchase - dependent variable
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(
    ["No Purchase (False)", "Purchase (True)"],
    counts.values,
    color=["#4575b4", "#d73027"],
    width=0.5,
    edgecolor="black",
    alpha=0.85,
)
for bar in bars:
    h = bar.get_height()
    ax.annotate(
        f"{h:,}\n({h/len(df):.1%})",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax.set_ylim(0, 12500)
ax.set_ylabel("Number of Sessions")
ax.set_title("Target Distribution: Revenue", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rf_class_distribution.png"), dpi=200)
plt.close()
print(f"Saved distribution plot to: {FIGURES_DIR}/rf_class_distribution.png")

Saved distribution plot to: results\figures/rf_class_distribution.png


In [59]:
# TRAIN-TEST

data = df.drop_duplicates().reset_index(drop=True)
data["Weekend"] = data["Weekend"].astype(int)
data["Revenue"] = data["Revenue"].astype(int)
print(f"Rows after removing duplicates: {len(data):,}")

numeric_features = [
        "Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
        "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
        "PageValues", "SpecialDay"
    ]
categorical_features = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
binary_features = ["Weekend"]

X = data[numeric_features + categorical_features + binary_features]
y = data["Revenue"]

Rows after removing duplicates: 12,205


In [60]:
RANDOM_STATE = 42
TEST_SIZE = 0.2


X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

print(f"Training partition: {X_train.shape[0]:,} sessions (Purchase rate = {y_train.mean():.4f})")
print(f"Test partition:     {X_test.shape[0]:,} sessions (Purchase rate = {y_test.mean():.4f})")

    # ColumnTransformer Preprocessor
numeric_pipeline = Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])

preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("bin", "passthrough", binary_features),
    ])

Training partition: 9,764 sessions (Purchase rate = 0.1563)
Test partition:     2,441 sessions (Purchase rate = 0.1565)
